# CDC PLACES Data Collection

This notebook pulls data from the CDC PLACES (Population Level Analysis and Community Estimates) API using the `CDCPLACES` R package.

## About CDC PLACES

PLACES provides model-based, population-level analysis and community estimates of health measures to all counties, places (incorporated and census designated places), census tracts, and ZIP Code Tabulation Areas (ZCTAs) across the United States.

**Source:** [CDCPLACES Package Documentation](https://brendensm.github.io/CDCPLACES/)

In [ ]:
# Install and load required packages
# Uncomment the install line if you need to install the package
# install.packages("CDCPLACES")
# Or install from GitHub: devtools::install_github("brendensm/CDCPLACES")
library(CDCPLACES)
library(dplyr)
library(ggplot2)

Installing package into ‘/Users/allisonlonderee/Library/R/arm64/4.5/library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘proxy’, ‘e1071’, ‘wk’, ‘classInt’, ‘s2’, ‘units’, ‘yyjsonr’, ‘tigris’, ‘sf’, ‘zctaCrosswalk’





The downloaded binary packages are in
	/var/folders/8v/kmqwjstx6sv9d9t9mh2qqpvw0000gn/T//RtmpY4KdJf/downloaded_packages


Warning message:
“package ‘CDCPLACES’ was built under R version 4.5.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




## Step 1: Explore Available Measures

First, let's get a dictionary of all available measures in the PLACES dataset.

In [5]:
# Get dictionary of available measures
measures_dict <- get_dictionary()

# Display first few rows
head(measures_dict, 40)

# View structure
str(measures_dict)

# Count total measures
cat("Total number of available measures:", nrow(measures_dict), "\n")

,measureid,measure_full_name,measure_short_name,categoryid,category_name,places_release_2024,measurename16_23,places_release_2023,places_release_2022,places_release_2021,places_release_2020,_500_cities_release_2019,_500_cities_release_2018,_500_cities_release_2017,_500_cities_release_2016,frequency_brfss_year,shortname16_23
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ARTHRITIS,Arthritis among adults,Arthritis,HLTHOUT,Health Outcomes,2022,Arthritis among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,NA
2,BPHIGH,High blood pressure among adults,High Blood Pressure,HLTHOUT,Health Outcomes,2021,High blood pressure among adults aged >=18 years,2021,2019,2019,2017,2017,2015,2015,2013,Odd,NA
3,CANCER,Cancer (non-skin) or melanoma among adults,Cancer (non-skin) or melanoma,HLTHOUT,Health Outcomes,2022,Cancer (excluding skin cancer) among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,Cancer (except skin)
4,CASTHMA,Current asthma among adults,Current Asthma,HLTHOUT,Health Outcomes,2022,Current asthma among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,NA
5,CHD,Coronary heart disease among adults,Coronary Heart Disease,HLTHOUT,Health Outcomes,2022,Coronary heart disease among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,NA
6,COPD,Chronic obstructive pulmonary disease among adults,COPD,HLTHOUT,Health Outcomes,2022,Chronic obstructive pulmonary disease among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,NA
7,DEPRESSION,Depression among adults,Depression,HLTHOUT,Health Outcomes,2022,Depression among adults aged >=18 years,2021,2020,2019,X,X,X,X,X,Every,NA
8,DIABETES,Diagnosed diabetes among adults,Diabetes,HLTHOUT,Health Outcomes,2022,Diagnosed diabetes among adults aged >=18 years,2021,2020,2019,2018,2017,2016,2015,2014,Every,NA
9,HIGHCHOL,High cholesterol among adults who have ever been screened,High Cholesterol,HLTHOUT,Health Outcomes,2021,High cholesterol among adults aged >=18 years who have been screened in the past 5 years,2021,2019,2019,2017,2017,2015,2015,2013,Odd,NA


'data.frame':	44 obs. of  17 variables:
 $ measureid               : chr  "ARTHRITIS" "BPHIGH" "CANCER" "CASTHMA" ...
 $ measure_full_name       : chr  "Arthritis among adults" "High blood pressure among adults" "Cancer (non-skin) or melanoma among adults" "Current asthma among adults" ...
 $ measure_short_name      : chr  "Arthritis" "High Blood Pressure" "Cancer (non-skin) or melanoma" "Current Asthma" ...
 $ categoryid              : chr  "HLTHOUT" "HLTHOUT" "HLTHOUT" "HLTHOUT" ...
 $ category_name           : chr  "Health Outcomes" "Health Outcomes" "Health Outcomes" "Health Outcomes" ...
 $ places_release_2024     : chr  "2022" "2021" "2022" "2022" ...
 $ measurename16_23        : chr  "Arthritis among adults aged >=18 years" "High blood pressure among adults aged >=18 years" "Cancer (excluding skin cancer) among adults aged >=18 years" "Current asthma among adults aged >=18 years" ...
 $ places_release_2023     : chr  "2021" "2021" "2021" "2021" ...
 $ places_release_2022     : c

## Step 2: Query PLACES Data

The `get_places()` function allows you to query data by:
- **Geography type**: county, census tract, ZCTA (ZIP Code), or place
- **State**: State abbreviation (e.g., "CA", "NY")
- **Measure**: Specific health measure ID from the dictionary
- **Release**: Data release year
- **Geometry**: Whether to include shapefiles (TRUE/FALSE)

### Example 1: Get county-level data for a specific state

In [10]:
# Example: Get county-level data for California
# Replace "CA" with your desired state abbreviation
# Replace "MeasureID" with an actual measure ID from the dictionary

# Example: Get county-level data for California
# Replace "CA" and measure_id below with your desired state and measure.

# --- Step 1: Find relevant measures ---
# PLACES does not include pesticide-specific measures. For pesticide exposure
# research, common proxies are respiratory outcomes (e.g., asthma, COPD).
# Filter by searching in measure names (columns: measure_full_name, measure_short_name):
relevant_measures <- measures_dict %>%
  filter(
    grepl("asthma|COPD|respiratory|disease|health|prevalence",
          measure_full_name, ignore.case = TRUE) |
    grepl("asthma|COPD|respiratory|disease|health",
          measure_short_name, ignore.case = TRUE)
  )

# Show matching measures and their IDs (use measureid when calling get_places)
print("Example measures relevant to environmental / respiratory health:")
relevant_measures %>%
  select(measureid, measure_short_name, measure_full_name) %>%
  head(20)

# --- Step 2: Fetch county-level data for one measure and state ---
state_abbr <- "CA"        # California; use any state abbreviation
measure_id <- "CASTHMA"   # Current asthma among adults (from dictionary)

ca_county_data <- get_places(
  geography = "county",
  state = state_abbr,
  measure = measure_id,
  release = 2023,
  geometry = FALSE
)

# --- Step 3: Inspect the result ---
cat("\nCounty-level data for", measure_id, "in", state_abbr, ":\n")
cat("Rows:", nrow(ca_county_data), "\n")
head(ca_county_data)

[1] "Example measures relevant to environmental / respiratory health:"


,measureid,measure_short_name,measure_full_name
,<chr>,<chr>,<chr>
1,CASTHMA,Current Asthma,Current asthma among adults
2,CHD,Coronary Heart Disease,Coronary heart disease among adults
3,COPD,COPD,Chronic obstructive pulmonary disease among adults
4,KIDNEY,Chronic Kidney Disease,Chronic kidney disease among adults aged >=18 years
5,GHLTH,General Health,Fair or poor self-rated health status among adults
6,ACCESS2,Health Insurance,Lack of health insurance among adults aged 18–64 years



County-level data for CASTHMA in CA :
Rows: 116 


,year,stateabbr,statedesc,locationname,datasource,category,measure,data_value_unit,data_value_type,data_value,low_confidence_limit,high_confidence_limit,totalpopulation,locationid,categoryid,measureid,datavaluetypeid,short_question_text,geolocation
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>
1,2021,CA,California,Calaveras,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Age-adjusted prevalence,10.3,9.0,11.7,46221,06009,HLTHOUT,CASTHMA,AgeAdjPrv,Current Asthma,"Point , -120.5541065, 38.1910682"
2,2021,CA,California,San Francisco,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Crude prevalence,8.2,7.2,9.3,815201,06075,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,"Point , -123.0322294, 37.7272391"
3,2021,CA,California,Los Angeles,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Age-adjusted prevalence,9.2,8.1,10.4,9829544,06037,HLTHOUT,CASTHMA,AgeAdjPrv,Current Asthma,"Point , -118.2618616, 34.1963983"
4,2021,CA,California,Calaveras,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Crude prevalence,9.9,8.8,11.2,46221,06009,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,"Point , -120.5541065, 38.1910682"
5,2021,CA,California,Tuolumne,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Crude prevalence,9.6,8.4,10.9,55810,06109,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,"Point , -119.9647335, 38.0214344"
6,2021,CA,California,Tulare,BRFSS,Health Outcomes,Current asthma among adults aged >=18 years,%,Age-adjusted prevalence,10.3,9.0,11.7,477054,06107,HLTHOUT,CASTHMA,AgeAdjPrv,Current Asthma,"Point , -118.7810551, 36.2288339"


### Example 2: Query specific measures for counties

In [13]:
# Example query structure (uncomment and modify as needed)
# Replace STATE_ABBR with your state (e.g., "CA", "NY", "TX")
# Replace MEASURE_ID with a measure ID from the dictionary above

places_data <- get_places(
  geography = "county",
  state = "OH",  # e.g., "CA" for California
  measure = "COPD",  # Replace with actual measure ID
  release = 2023,  # Check available releases
  geometry = FALSE  # Set to TRUE if you need shapefiles
)
# View the data
head(places_data)
str(places_data)

,year,stateabbr,statedesc,locationname,datasource,category,measure,data_value_unit,data_value_type,data_value,low_confidence_limit,high_confidence_limit,totalpopulation,locationid,categoryid,measureid,datavaluetypeid,short_question_text,geolocation
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>
1,2021,OH,Ohio,Jefferson,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,11.1,9.4,13.0,64789,39081,HLTHOUT,COPD,CrdPrv,COPD,"Point , -80.7635272, 40.3993836"
2,2021,OH,Ohio,Portage,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,8.0,6.6,9.6,162382,39133,HLTHOUT,COPD,CrdPrv,COPD,"Point , -81.1969618, 41.1689703"
3,2021,OH,Ohio,Pike,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,10.7,8.9,12.6,27089,39131,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -83.0529107, 39.0713413"
4,2021,OH,Ohio,Gallia,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,11.0,9.2,13.0,29158,39053,HLTHOUT,COPD,CrdPrv,COPD,"Point , -82.3017463, 38.8170456"
5,2021,OH,Ohio,Hocking,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,9.1,7.5,10.7,28097,39073,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -82.4834473, 39.4903424"
6,2021,OH,Ohio,Franklin,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,6.7,5.5,7.9,1321414,39049,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -83.0090858, 39.9698749"


'data.frame':	176 obs. of  19 variables:
 $ year                 : chr  "2021" "2021" "2021" "2021" ...
 $ stateabbr            : chr  "OH" "OH" "OH" "OH" ...
 $ statedesc            : chr  "Ohio" "Ohio" "Ohio" "Ohio" ...
 $ locationname         : chr  "Jefferson" "Portage" "Pike" "Gallia" ...
 $ datasource           : chr  "BRFSS" "BRFSS" "BRFSS" "BRFSS" ...
 $ category             : chr  "Health Outcomes" "Health Outcomes" "Health Outcomes" "Health Outcomes" ...
 $ measure              : chr  "Chronic obstructive pulmonary disease among adults aged >=18 years" "Chronic obstructive pulmonary disease among adults aged >=18 years" "Chronic obstructive pulmonary disease among adults aged >=18 years" "Chronic obstructive pulmonary disease among adults aged >=18 years" ...
 $ data_value_unit      : chr  "%" "%" "%" "%" ...
 $ data_value_type      : chr  "Crude prevalence" "Crude prevalence" "Age-adjusted prevalence" "Crude prevalence" ...
 $ data_value           : num  11.1 8 10.7 11 9.1 6

### Example 3: Query multiple measures or geographies

You can query multiple measures by calling `get_places()` multiple times and combining the results.

In [15]:
# Example: Query multiple measures for the same geography
# This is a template - customize based on your needs

measure_ids <- c("COPD", "CASTHMA")

all_data <- lapply(measure_ids, function(measure_id) {
  get_places(
    geography = "county",
    state = "OH",
    measure = measure_id,
    release = 2023,
    geometry = FALSE
  )
})

# Combine results
combined_data <- bind_rows(all_data)
head(combined_data)

,year,stateabbr,statedesc,locationname,datasource,category,measure,data_value_unit,data_value_type,data_value,low_confidence_limit,high_confidence_limit,totalpopulation,locationid,categoryid,measureid,datavaluetypeid,short_question_text,geolocation
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<list>
1,2021,OH,Ohio,Jefferson,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,11.1,9.4,13.0,64789,39081,HLTHOUT,COPD,CrdPrv,COPD,"Point , -80.7635272, 40.3993836"
2,2021,OH,Ohio,Portage,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,8.0,6.6,9.6,162382,39133,HLTHOUT,COPD,CrdPrv,COPD,"Point , -81.1969618, 41.1689703"
3,2021,OH,Ohio,Pike,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,10.7,8.9,12.6,27089,39131,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -83.0529107, 39.0713413"
4,2021,OH,Ohio,Gallia,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Crude prevalence,11.0,9.2,13.0,29158,39053,HLTHOUT,COPD,CrdPrv,COPD,"Point , -82.3017463, 38.8170456"
5,2021,OH,Ohio,Hocking,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,9.1,7.5,10.7,28097,39073,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -82.4834473, 39.4903424"
6,2021,OH,Ohio,Franklin,BRFSS,Health Outcomes,Chronic obstructive pulmonary disease among adults aged >=18 years,%,Age-adjusted prevalence,6.7,5.5,7.9,1321414,39049,HLTHOUT,COPD,AgeAdjPrv,COPD,"Point , -83.0090858, 39.9698749"


## Step 3: Data Exploration and Analysis

Once you have the data, you can explore and analyze it.

In [ ]:
# Example data exploration (uncomment when you have data)
# 
# # Summary statistics
# summary(places_data)
# 
# # Check for missing values
# colSums(is.na(places_data))
# 
# # Basic visualization
# # ggplot(places_data, aes(x = Data_Value)) +
# #   geom_histogram(bins = 30) +
# #   labs(title = "Distribution of Health Measure Values",
# #        x = "Data Value",
# #        y = "Frequency")

## Next Steps

1. **Identify relevant measures**: Review the measures dictionary to find health measures related to your research question
2. **Select geography**: Choose the appropriate geography level (county, census tract, ZCTA, or place)
3. **Query data**: Use `get_places()` to pull the data for your selected measures and geographies
4. **Clean and analyze**: Process the data as needed for your analysis

## Resources

- [CDCPLACES Package Documentation](https://brendensm.github.io/CDCPLACES/)
- [CDC PLACES Website](https://www.cdc.gov/places/)
- [Package Vignette](https://brendensm.github.io/CDCPLACES/articles/CDCPLACES.html)